# 05 — hERG GCN pilot: scaffold-aware, CPU-first

**Goal:** compare a simple molecular GCN with the already evaluated RF-Hybrid and SVM-Hybrid on the **exact same original 6,841 training molecules and three StratifiedGroupKFold folds**. Do not use the 1,074-molecule dev set or the 2,085-molecule locked test for model design.

This notebook is a **research pilot, not yet a valid submission**. Neural-network to ONNX conversion, fixed-size `featurize`, and permitted package versions must be checked separately against `Part1_TEMPLATE.py` / `TestModel.py` before any submission. The GCN uses **vanilla PyTorch** (no torch-geometric).

**Runtime limit:** start with `EPOCHS=20`, `PATIENCE=5`, 3 outer folds, and one architecture. Compare fold-wise AUROC/AUPRC first, not the previously seen development AUROC.

In [ ]:
from pathlib import Path
import copy, time, random, sys
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
RDLogger.DisableLog('rdApp.warning')
try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
except ImportError as exc:
    raise ImportError('PyTorch is required for the research-only GCN pilot. Check your course environment and its package restrictions before installing.') from exc

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE=torch.device('cpu')  # consistent with coursework CPU constraint
OUT=Path('research_outputs/gcn_pilot'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('hw1data.tab')
assert DATA.exists(), 'Run from the CDD203-HW1 directory alongside hw1data.tab'
print('Python:',sys.version.split()[0], '| PyTorch:',torch.__version__, '| device:',DEVICE)

## 1. Reproduce original split and folds exactly

The split construction below matches Notebook 04. **Only** `train_idx` is used for GCN training and fold evaluation.

In [ ]:
df=pd.read_csv(DATA,sep='\t')
assert {'Drug','Y'}.issubset(df.columns)
smiles=df['Drug'].tolist(); mols=[Chem.MolFromSmiles(s) for s in smiles]
assert all(m is not None for m in mols)
y=df['Y'].to_numpy(dtype=np.int64)
groups=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m),canonical=True) for m in mols])
idx=np.arange(len(df))
trainval_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(idx,y,groups))
a,c=next(GroupShuffleSplit(n_splits=1,test_size=.125,random_state=SEED).split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[a];val_idx=trainval_idx[c]
assert (len(train_idx),len(val_idx),len(test_idx))==(6841,1074,2085)
assert set(groups[train_idx]).isdisjoint(groups[val_idx])
assert set(groups[train_idx]).isdisjoint(groups[test_idx])
y_train=y[train_idx];g_train=groups[train_idx];train_mols=[mols[i] for i in train_idx]
outer_cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)
outer_folds=list(outer_cv.split(np.zeros(len(train_idx)),y_train,g_train))
assert all(not(set(g_train[tr])&set(g_train[va])) for tr,va in outer_folds)
print('TRAIN:',len(train_idx),'DEV (untouched):',len(val_idx),'LOCKED TEST (untouched):',len(test_idx))
print('Outer fold validation sizes:',[len(va) for tr,va in outer_folds])

## 2. Atom features and graph construction

Each molecule is converted into a graph with 16 numeric atom features and a normalized self-loop adjacency matrix. Padding + masks allow batching variable-size graphs. **Mean pooling excludes padding**, and all graph construction occurs before model inference (so it must later be included in speed measurements). This is a basic GCN, not an edge-aware molecular GNN; bond types are deliberately not encoded in this pilot.

In [ ]:
ATOM_TYPES=[1,6,7,8,9,15,16,17,35,53]
ATOM_FDIM=len(ATOM_TYPES)+6

def molecule_to_graph(m):
    atoms=list(m.GetAtoms()); nnodes=len(atoms)
    if nnodes==0: raise ValueError('Empty molecular graph')
    x=np.zeros((nnodes,ATOM_FDIM),dtype=np.float32)
    for i,atom in enumerate(atoms):
        z=atom.GetAtomicNum()
        if z in ATOM_TYPES: x[i,ATOM_TYPES.index(z)]=1.0
        # 6 scaled numerical / binary features
        x[i,len(ATOM_TYPES):]=[
            min(atom.GetTotalDegree(),5)/5,
            max(-3,min(3,atom.GetFormalCharge()))/3,
            float(atom.GetIsAromatic()),
            float(atom.IsInRing()),
            min(atom.GetTotalNumHs(),4)/4,
            min(atom.GetHybridization()==Chem.rdchem.HybridizationType.SP3,1)
        ]
    adjacency=np.eye(nnodes,dtype=np.float32)
    for bond in m.GetBonds():
        i=bond.GetBeginAtomIdx();j=bond.GetEndAtomIdx()
        adjacency[i,j]=1;adjacency[j,i]=1
    deg=adjacency.sum(axis=1)
    inv=np.power(deg,-.5)
    adjacency=inv[:,None]*adjacency*inv[None,:]
    return x,adjacency

class MolGraphDataset(Dataset):
    def __init__(self,molecules,labels):
        self.graphs=[molecule_to_graph(m) for m in molecules]
        self.labels=np.asarray(labels,dtype=np.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i): return self.graphs[i][0],self.graphs[i][1],self.labels[i]

def collate_graph(batch):
    b=len(batch);sizes=[v[0].shape[0] for v in batch];maxn=max(sizes)
    x=torch.zeros((b,maxn,ATOM_FDIM),dtype=torch.float32)
    adj=torch.zeros((b,maxn,maxn),dtype=torch.float32)
    mask=torch.zeros((b,maxn,1),dtype=torch.float32)
    yb=torch.zeros(b,dtype=torch.float32)
    for i,(xi,ai,yi) in enumerate(batch):
        k=len(xi)
        x[i,:k]=torch.from_numpy(xi)
        adj[i,:k,:k]=torch.from_numpy(ai)
        mask[i,:k]=1
        yb[i]=float(yi)
    return x,adj,mask,yb

class GCN(nn.Module):
    def __init__(self,dim=64,dropout=.20):
        super().__init__()
        self.conv1=nn.Linear(ATOM_FDIM,dim)
        self.conv2=nn.Linear(dim,dim)
        self.dropout=nn.Dropout(dropout)
        self.head=nn.Sequential(nn.Linear(dim,dim//2),nn.ReLU(),nn.Linear(dim//2,1))
    def forward(self,x,adj,mask):
        h=torch.relu(self.conv1(torch.bmm(adj,x)))
        h=self.dropout(h)
        h=torch.relu(self.conv2(torch.bmm(adj,h)))
        pooled=(h*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
        return self.head(pooled).squeeze(-1)

train_ds=MolGraphDataset(train_mols,y_train)
print('Graph dataset:',len(train_ds),'atom feature dim:',ATOM_FDIM,
      'max atoms:',max(graph[0].shape[0] for graph in train_ds.graphs))

## 3. Precommitted training settings

Use a small **inner scaffold-disjoint holdout only inside the outer training fold** for early stopping. The outer validation fold never determines epoch count or model parameters. Only one architecture is evaluated in this pilot.

In [ ]:
EPOCHS=20
PATIENCE=5
BATCH_SIZE=64
LEARNING_RATE=1e-3
WEIGHT_DECAY=1e-4
HIDDEN_DIM=64
DROPOUT=.20

@torch.no_grad()
def predict_prob(model,loader):
    model.eval();pp=[];yy=[]
    for x,adj,mask,yb in loader:
        logits=model(x.to(DEVICE),adj.to(DEVICE),mask.to(DEVICE))
        pp.append(torch.sigmoid(logits).cpu().numpy());yy.append(yb.numpy())
    return np.concatenate(yy).astype(int),np.concatenate(pp)

def train_one_fold(tr,va,fold_id):
    # Early-stop split uses ONLY outer training records
    inner=GroupShuffleSplit(n_splits=1,test_size=.12,random_state=SEED+fold_id)
    i_tr,i_stop=next(inner.split(tr,y_train[tr],g_train[tr]))
    fit_indices=tr[i_tr];stop_indices=tr[i_stop]
    assert not(set(g_train[fit_indices])&set(g_train[stop_indices]))
    assert not(set(g_train[tr])&set(g_train[va]))

    train_loader=DataLoader(torch.utils.data.Subset(train_ds,fit_indices.tolist()),batch_size=BATCH_SIZE,
                           shuffle=True,collate_fn=collate_graph,
                           generator=torch.Generator().manual_seed(SEED+fold_id))
    stop_loader=DataLoader(torch.utils.data.Subset(train_ds,stop_indices.tolist()),batch_size=BATCH_SIZE,
                          shuffle=False,collate_fn=collate_graph)
    eval_loader=DataLoader(torch.utils.data.Subset(train_ds,va.tolist()),batch_size=BATCH_SIZE,
                          shuffle=False,collate_fn=collate_graph)
    torch.manual_seed(SEED+fold_id)
    net=GCN(dim=HIDDEN_DIM,dropout=DROPOUT).to(DEVICE)
    opt=torch.optim.Adam(net.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    loss_fn=nn.BCEWithLogitsLoss()
    best_state=None;best_loss=float('inf');bad=0;best_epoch=0
    history=[]
    start=time.perf_counter()
    for epoch in range(1,EPOCHS+1):
        net.train()
        train_total=0.; train_count=0
        for xb,ab,mb,yb in train_loader:
            opt.zero_grad()
            logits=net(xb.to(DEVICE),ab.to(DEVICE),mb.to(DEVICE))
            loss=loss_fn(logits,yb.to(DEVICE))
            loss.backward();opt.step()
            train_total+=loss.detach().item()*len(yb);train_count+=len(yb)
        train_loss=train_total/train_count
        net.eval();total=0.;count=0
        with torch.no_grad():
            for xb,ab,mb,yb in stop_loader:
                logits=net(xb.to(DEVICE),ab.to(DEVICE),mb.to(DEVICE))
                loss=loss_fn(logits,yb.to(DEVICE)).item()
                total+=loss*len(yb);count+=len(yb)
        stop_loss=total/count
        history.append({"Fold":fold_id,"Epoch":epoch,"TrainLoss":train_loss,"StopLoss":stop_loss})
        if stop_loss < best_loss-1e-4:
            best_loss=stop_loss;best_epoch=epoch;bad=0
            best_state=copy.deepcopy(net.state_dict())
        else:
            bad+=1
            if bad>=PATIENCE:break
    net.load_state_dict(best_state)
    yt,prob=predict_prob(net,eval_loader)
    assert np.array_equal(yt,y_train[va])
    return net,prob,best_epoch,time.perf_counter()-start,history


## 4. Run the three scaffold-aware outer folds (CPU)

The cell prints incremental fold results and saves predictions so the next cells can be rerun without retraining.

In [ ]:
oof_gcn=np.full(len(y_train),np.nan,dtype=float)
fold_rows=[]
all_histories=[]
for fold_id,(tr,va) in enumerate(outer_folds,1):
    net,prob,best_epoch,seconds,history=train_one_fold(tr,va,fold_id)
    all_histories.extend(history)
    oof_gcn[va]=prob
    row={'Model':'GCN-2layer','Fold':fold_id,'n_validation':len(va),
         'AUROC':roc_auc_score(y_train[va],prob),
         'AUPRC':average_precision_score(y_train[va],prob),
         'BestEpoch':best_epoch,'TrainingSeconds':seconds}
    fold_rows.append(row)
    print('Fold',fold_id,'AUROC',round(row['AUROC'],4),'AUPRC',round(row['AUPRC'],4),
          'epoch',best_epoch,'train_s',round(seconds,1))
assert np.isfinite(oof_gcn).all()
fold_metrics=pd.DataFrame(fold_rows)
display(fold_metrics.round(4))
print('Mean fold AUROC',round(fold_metrics.AUROC.mean(),4),
      '| SD',round(fold_metrics.AUROC.std(ddof=1),4),
      '| pooled OOF AUROC',round(roc_auc_score(y_train,oof_gcn),4))
fold_metrics.to_csv(OUT/'gcn_fold_metrics.csv',index=False)
pd.DataFrame({'OriginalRow':train_idx,'Y':y_train,'GCN_OOF_Probability':oof_gcn}).to_csv(
    OUT/'gcn_oof_predictions.csv',index=False)
loss_history=pd.DataFrame(all_histories)
loss_history.to_csv(OUT/"gcn_epoch_loss_history.csv",index=False)
print("Saved:",OUT/"gcn_epoch_loss_history.csv")


## 4A. Diagnostic: training versus inner early-stopping loss

Training loss is measured during each epoch in training mode (with dropout); stopping loss is measured afterward without dropout on a scaffold-disjoint subset of outer TRAIN. Compare trends, not absolute gaps. The outer validation folds and locked test are never used for early stopping.

In [ ]:
from IPython.display import display, SVG

assert "loss_history" in globals(), "Run section 4 training first."
display(loss_history.groupby("Fold").agg(
    TrainedEpochs=("Epoch", "max"),
    LastTrainLoss=("TrainLoss", "last"),
    LastStopLoss=("StopLoss", "last"),
    BestStopLoss=("StopLoss", "min")
).round(4))

for fold_id, group in loss_history.groupby("Fold"):
    width, height = 700, 250
    group = group.sort_values("Epoch")
    e_min, e_max = int(group.Epoch.min()), int(group.Epoch.max())
    v = np.r_[group.TrainLoss.to_numpy(), group.StopLoss.to_numpy()]
    low, high = float(v.min()), float(v.max())
    pad = max(.01, (high-low)*.12)
    low, high = low-pad, high+pad
    def xcoord(e): return 65+(float(e)-e_min)/max(1,e_max-e_min)*600
    def ycoord(y): return 205-(float(y)-low)/(high-low)*160
    paths = []
    for col, color in [("TrainLoss", "#2563eb"), ("StopLoss", "#dc2626")]:
        points = " ".join(f"{xcoord(e):.1f},{ycoord(y):.1f}" for e,y in zip(group.Epoch,group[col]))
        paths.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2.5" />')
    best_epoch = int(group.loc[group.StopLoss.idxmin(), "Epoch"])
    best_x = xcoord(best_epoch)
    chart = (f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">'
             f'<rect width="100%" height="100%" fill="white" />'
             f'<text x="65" y="20" fill="black">Fold {fold_id}: loss by epoch</text>'
             f'<text x="370" y="20" fill="#2563eb">Train</text>'
             f'<text x="460" y="20" fill="#dc2626">Early stopping</text>'
             f'<line x1="65" x2="65" y1="35" y2="205" stroke="black" />'
             f'<line x1="65" x2="665" y1="205" y2="205" stroke="black" />'
             f'<line x1="{best_x:.1f}" x2="{best_x:.1f}" y1="35" y2="205" stroke="gray" stroke-dasharray="4,4" />'
             + "".join(paths)
             + f'<text x="65" y="235" fill="black">Epoch {e_min}–{e_max}; dotted line: best epoch {best_epoch}</text>'
             + f'<text x="5" y="45" fill="black">{high:.3f}</text><text x="5" y="205" fill="black">{low:.3f}</text></svg>')
    display(SVG(chart))

print("Loss table:", OUT/"gcn_epoch_loss_history.csv")

## 5. Compare with matched RF/SVM cross-validation

Paste the *matched fold-wise* RF/SVM AUROC and AUPRC from Notebook 04 if desired, or read its saved predictions if you have exported them. **Do not compare an OOF score directly to a development-validation score**. The model comparison is descriptive: GCN is not part of a nested model selection experiment.

In [ ]:
# Cross-validation reference from Notebook 04 (same original TRAIN / same folds):
# Fill with actual per-fold results if available; no guessed RF/SVM values.
comparison=pd.DataFrame({
    'Model':['GCN-2layer'],
    'MeanFoldAUROC':[fold_metrics.AUROC.mean()],
    'SDFoldAUROC':[fold_metrics.AUROC.std(ddof=1)],
    'MeanFoldAUPRC':[fold_metrics.AUPRC.mean()],
    'PooledOOFAUROC':[roc_auc_score(y_train,oof_gcn)]
})
display(comparison.round(4))
print('For matched RF/SVM scores, re-open Notebook 04 and use its OOF outputs,')
print('or compare with previously reported scaffold-aware RF/SVM results, clearly labelled.')

## 6. CPU GCN inference timing, including graph construction

Use the same **original TRAIN SMILES** as the GCN validation folds (rather than a dev or locked test subset). We time (a) RDKit graph generation + padding and (b) Torch prediction separately. A batch's max atom count influences timing. This is not yet an ONNX benchmark.

In [ ]:
# Retrain one fold only if not already run; uses the last net from cell 4.
assert 'net' in globals(), 'Run the 3-fold training cell above first.'
sample_indices=outer_folds[-1][1]
sample_smiles=[smiles[train_idx[i]] for i in sample_indices]
N_REPEATS=3
bench=[]
for repeat in range(N_REPEATS):
    start=time.perf_counter()
    graphs=[molecule_to_graph(Chem.MolFromSmiles(s)) for s in sample_smiles]
    graph_seconds=time.perf_counter()-start
    tmp_ds=type('GraphListDataset',(Dataset,),{
        '__len__':lambda self:len(graphs),
        '__getitem__':lambda self,i:(graphs[i][0],graphs[i][1],0.)})()
    loader=DataLoader(tmp_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_graph)
    start=time.perf_counter()
    _,p=predict_prob(net,loader)
    inference_seconds=time.perf_counter()-start
    bench.append({'Repeat':repeat+1,'Molecules':len(sample_smiles),
                  'GraphConstructionSeconds':graph_seconds,
                  'TorchPredictionSeconds':inference_seconds,
                  'TotalSeconds':graph_seconds+inference_seconds,
                  'MoleculesPerSecond':len(sample_smiles)/(graph_seconds+inference_seconds)})
speed_df=pd.DataFrame(bench)
display(speed_df.median(numeric_only=True).to_frame('Median').T.round(4))
speed_df.to_csv(OUT/'gcn_cpu_speed.csv',index=False)
print('For a strict RF/SVM comparison, benchmark the SAME SMILES and batch size for all models.')

## 7. Decision gate

- If GCN is **worse or tied** on matched scaffold CV and is slower/less deployable, retain the RF/SVM candidates.
- If GCN consistently improves held-out folds, verify robustness and inspect ONNX feasibility **before** recommending it for submission.
- Do not tune to development or locked-test results.
- The assignment requires fixed-size 2D `featurize` and an ONNX model; this variable-node implementation **cannot be submitted as-is**.

**Send back:** `fold_metrics`, `comparison`, and the speed table. We will then decide whether graph+fingerprint fusion deserves a separate, limited experiment.